In [ ]:
import sys
from pathlib import Path

def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / ".git").exists():
            return p
    raise RuntimeError("Could not locate project root (no .git found above cwd)")

PROJECT_ROOT = _find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments" / "mlp" / "experience_replay"


# MLP Experience Replay
Train an MLPClassifier with year-wise incremental scaling and experience replay.

In [ ]:
import copy
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm

from src.mlp_replay.data import (
    load_dataset_and_splits,
    prepare_raw_features_for_year,
    prepare_features_for_year,
    precompute_yearly_raw_cache,
)
from src.mlp_replay.model import (
    build_mlp_model,
    capture_model_state,
    restore_model_state,
)
from src.mlp_replay.replay_strategies import (
    build_replay_year_spans,
    materialize_replay_samples_in_order,
)
from src.mlp_replay.training_loop import (
    resume_or_init_ratio_state,
    load_year_batch_or_none,
    finalize_ratio_and_save,
    write_combined_history_csv,
    merge_all_ratio_histories,
)
from src.mlp_replay.checkpointing import (
    create_empty_training_history,
    load_all_training_histories,
    save_all_training_histories,
    load_completion_status,
    save_completion_status,
    append_training_log,
    format_ratio_key,
    get_cached_raw_year,
    per_ratio_path,
)

warnings.filterwarnings('ignore')

## 1. Load Data and Splits

In [ ]:
ds_path = PROJECT_ROOT / 'training_data_with_features_plus_monthly_indices.zarr'
split_path = PROJECT_ROOT / 'data_split.npz'
ds, train_pixel_indices, val_pixel_indices, test_pixel_indices = load_dataset_and_splits(ds_path, split_path)


## 2. Feature Engineering

`prepare_raw_features_for_year` and `prepare_features_for_year` now live in `src/mlp_replay/data.py` and are imported above.


## 3. Initialize Class Weights and MLP

In [ ]:
print('Precomputing raw yearly features for train/validation splits...')
n_years = len(ds.year)


train_feature_cache = precompute_yearly_raw_cache(ds, train_pixel_indices, n_years, 'train', ds_path=ds_path)
val_feature_cache = precompute_yearly_raw_cache(ds, val_pixel_indices, n_years, 'validation', ds_path=ds_path)

print('Computing class weights from cached training labels...')
train_label_batches = [y_batch for _, y_batch in train_feature_cache.values() if len(y_batch) > 0]
if not train_label_batches:
    raise ValueError('No valid training labels after filtering.')

all_train_labels = np.concatenate(train_label_batches).astype(int)
all_train_labels = all_train_labels[np.isin(all_train_labels, [0, 1])]
if len(all_train_labels) == 0:
    raise ValueError('No valid training labels after filtering.')

classes = np.array([0, 1])
class_weights_array = compute_class_weight('balanced', classes=classes, y=all_train_labels)
class_weight_dict = {classes[i]: class_weights_array[i] for i in range(len(classes))}

print('Class weights:')
print(f"  Class 0: {class_weight_dict[0]:.4f}")
print(f"  Class 1: {class_weight_dict[1]:.4f}")

model = MLPClassifier(
    hidden_layer_sizes=(64,),
    activation='relu',
    alpha=0.0001,
    random_state=42,
    solver='adam',
    learning_rate='adaptive',
    max_iter=1,
    learning_rate_init=0.001,
    warm_start=False,
    verbose=False,
)
print('MLP initialized')

## 4. Replay-Enabled Online Training

In [ ]:
import json
from datetime import datetime

REPLAY_RATIOS = [0.2, 0.3, 0.4, 0.5]

import os
_replay_ratio_override = os.environ.get('MLP_REPLAY_RATIO_OVERRIDE')
if _replay_ratio_override:
    REPLAY_RATIOS = [float(_replay_ratio_override)]
REPLAY_ENABLED = True
REPLAY_RANDOM_STATE = 42

CHUNK_SIZE = 50000
MAX_EPOCHS = 15
PATIENCE = 3
MIN_DELTA = 0.0005

CHECKPOINT_DIR = EXPERIMENTS_DIR / 'training_checkpoints_mlp_experience_replay'
ALL_HISTORIES_FILE = CHECKPOINT_DIR / 'mlp_replay_all_training_histories.pkl'
COMPLETION_STATUS_FILE = CHECKPOINT_DIR / 'mlp_replay_completion_status.json'
TRAINING_LOG_FILE = CHECKPOINT_DIR / 'mlp_replay_training_log.txt'
COMBINED_HISTORY_FILENAME = 'mlp_classifier_history_prevyears_monthly_features_incremental_scaler_experience_replay_all_ratios.csv'

CHECKPOINT_DIR.mkdir(exist_ok=True)

if 'train_feature_cache' not in globals() or 'val_feature_cache' not in globals():
    raise ValueError('Raw feature caches not found. Run Cell 8 first to precompute yearly features.')


n_years = len(ds.year)
year_values = ds.year.values
year_value_to_idx = {int(y): idx for idx, y in enumerate(year_values)}
all_target_year_values = [int(year_values[idx]) for idx in range(1, n_years)]

print(f'Checkpoint directory: {CHECKPOINT_DIR.resolve()}')
print(f'Replay ratios: {REPLAY_RATIOS}')
append_training_log(TRAINING_LOG_FILE, f'Started training run for ratios: {REPLAY_RATIOS}')

for ratio_idx, replay_ratio in enumerate(REPLAY_RATIOS):
    ratio_key = format_ratio_key(replay_ratio)
    ratio_histories_file = per_ratio_path(ALL_HISTORIES_FILE, ratio_key)
    ratio_completion_file = per_ratio_path(COMPLETION_STATUS_FILE, ratio_key)
    ratio_log_file = per_ratio_path(TRAINING_LOG_FILE, ratio_key)
    all_training_histories = load_all_training_histories(ratio_histories_file)
    completion_status = load_completion_status(ratio_completion_file)
    output_suffix = f'incremental_scaler_experience_replay_{ratio_key}'
    models_dir_name = f'models_mlp_prevyears_monthly_features_{output_suffix}'
    scaler_file_template = f'scaler_year_{{year}}_mlp_prevyears_monthly_features_{output_suffix}.pkl'
    model_file_template = f'model_year_{{year}}_mlp_prevyears_monthly_features_{output_suffix}.pkl'
    final_scaler_filename = f'scaler_final_mlp_prevyears_monthly_features_{output_suffix}.pkl'
    final_model_filename = f'mlp_classifier_model_prevyears_monthly_features_{output_suffix}.pkl'
    history_filename = f'mlp_classifier_history_prevyears_monthly_features_{output_suffix}.csv'

    models_dir = EXPERIMENTS_DIR / models_dir_name
    models_dir.mkdir(exist_ok=True)

    if ratio_key in completion_status.get('completed_ratios', []):
        print(f'[{ratio_key}] already completed. Skipping ratio.')
        append_training_log(ratio_log_file, f'[{ratio_key}] skipped (already completed).')
        continue
    replay_rng = np.random.default_rng(REPLAY_RANDOM_STATE + ratio_idx)

    model, incremental_scaler, start_year_idx, completed_years, training_history = resume_or_init_ratio_state(
        ratio_key, models_dir, model_file_template, scaler_file_template, year_value_to_idx,
        all_training_histories, completion_status, ratio_log_file,
        extra_history_keys=None,
    )

    print(f'[{ratio_key}] Model output directory: {models_dir.resolve()}')
    print(f'[{ratio_key}] Years to train: 1..{n_years - 1} (year 0 skipped)')

    for year_idx in tqdm(range(start_year_idx, n_years), desc=f'{ratio_key} by year'):
        year_val = int(year_values[year_idx])

        if year_val in completed_years:
            continue

        year_batch = load_year_batch_or_none(
            train_feature_cache, val_feature_cache, year_idx, year_val, ratio_key,
            completed_years, completion_status, ratio_completion_file, ratio_log_file,
        )
        if year_batch is None:
            continue
        X_train_raw, y_train_batch, X_val_raw, y_val_batch = year_batch

        incremental_scaler.partial_fit(X_train_raw)
        X_train_batch = incremental_scaler.transform(X_train_raw)
        X_val_batch = incremental_scaler.transform(X_val_raw)

        scaler_checkpoint = copy.deepcopy(incremental_scaler)
        year_scaler_path = models_dir / scaler_file_template.format(year=year_val)
        with open(year_scaler_path, 'wb') as f:
            pickle.dump(scaler_checkpoint, f)

        n_samples = len(X_train_batch)
        replay_target_size = int(n_samples * replay_ratio) if REPLAY_ENABLED else 0

        # Spans only -- prior years' FEATURES are deliberately not concatenated
        # here. The scaled pool reached ~3.6GB at the last training year (~7.2GB
        # counting the np.vstack duplicate kept alive beside it) only to be
        # reduced to at most replay_used_size rows, and that is what the Kaggle
        # OOM killer was reacting to. The same rows are now reached one year at
        # a time; which rows get selected is unchanged.
        if REPLAY_ENABLED and year_idx > 1:
            replay_year_spans, y_replay_pool, replay_pool_size = build_replay_year_spans(
                lambda past_year_idx: get_cached_raw_year(train_feature_cache, past_year_idx)[1],
                year_idx,
            )
        else:
            replay_year_spans = []
            y_replay_pool = np.empty((0,), dtype=y_train_batch.dtype)
            replay_pool_size = 0

        def _load_raw_past_year(past_year_idx):
            return get_cached_raw_year(train_feature_cache, past_year_idx)

        replay_used_size = min(replay_target_size, replay_pool_size) if REPLAY_ENABLED else 0

        if hasattr(model, 'n_features_in_') and int(model.n_features_in_) != int(X_train_batch.shape[1]):
            raise ValueError(
                f'Feature count mismatch at year {year_val}: model expects {model.n_features_in_}, got {X_train_batch.shape[1]}'
            )

        # One-time yearly sampling for replay segments (no resampling per epoch).
        if replay_used_size > 0:
            replay_indices = replay_rng.choice(replay_pool_size, size=replay_used_size, replace=False)
            X_replay_sampled, y_replay_sampled = materialize_replay_samples_in_order(
                replay_indices, replay_year_spans, _load_raw_past_year, incremental_scaler,
            )
            X_combined_base = np.concatenate([X_train_batch, X_replay_sampled], axis=0)
            y_combined_base = np.concatenate([y_train_batch, y_replay_sampled], axis=0)
        else:
            X_combined_base = X_train_batch
            y_combined_base = y_train_batch

        best_val_pr_auc = -np.inf
        patience_counter = 0
        best_model_state = None

        for epoch in range(MAX_EPOCHS):
            combined_n_samples = len(X_combined_base)
            shuffle_idx = replay_rng.permutation(combined_n_samples)
            X_train_shuffled = X_combined_base[shuffle_idx]
            y_train_shuffled = y_combined_base[shuffle_idx]
            n_chunks = max(1, int(np.ceil(combined_n_samples / CHUNK_SIZE)))

            for chunk_idx in range(n_chunks):
                start_idx = chunk_idx * CHUNK_SIZE
                end_idx = min(start_idx + CHUNK_SIZE, combined_n_samples)
                X_chunk = X_train_shuffled[start_idx:end_idx]
                y_chunk = y_train_shuffled[start_idx:end_idx]
                sample_weights_chunk = np.array([class_weight_dict[int(label)] for label in y_chunk])
                model.partial_fit(X_chunk, y_chunk, classes=classes, sample_weight=sample_weights_chunk)

            y_val_pred = model.predict(X_val_batch)
            y_val_proba = model.predict_proba(X_val_batch)[:, 1]
            val_pr_auc = average_precision_score(y_val_batch, y_val_proba) if len(np.unique(y_val_batch)) > 1 else np.nan

            if val_pr_auc > best_val_pr_auc + MIN_DELTA:
                best_val_pr_auc = val_pr_auc
                patience_counter = 0
                best_model_state = capture_model_state(model)
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    break

        if best_model_state is not None:
            restore_model_state(model, best_model_state)

        y_train_pred = model.predict(X_train_batch)
        y_val_pred = model.predict(X_val_batch)
        y_val_proba = model.predict_proba(X_val_batch)[:, 1]

        train_acc = accuracy_score(y_train_batch, y_train_pred)
        train_prec = precision_score(y_train_batch, y_train_pred, zero_division=0)
        train_rec = recall_score(y_train_batch, y_train_pred, zero_division=0)
        train_f1 = f1_score(y_train_batch, y_train_pred, zero_division=0)

        val_acc = accuracy_score(y_val_batch, y_val_pred)
        val_prec = precision_score(y_val_batch, y_val_pred, zero_division=0)
        val_rec = recall_score(y_val_batch, y_val_pred, zero_division=0)
        val_f1 = f1_score(y_val_batch, y_val_pred, zero_division=0)
        if len(np.unique(y_val_batch)) > 1:
            val_roc_auc = roc_auc_score(y_val_batch, y_val_proba)
            val_pr_auc = average_precision_score(y_val_batch, y_val_proba)
        else:
            val_roc_auc = np.nan
            val_pr_auc = np.nan

        training_history['year'].append(year_val)
        training_history['train_accuracy'].append(train_acc)
        training_history['train_precision'].append(train_prec)
        training_history['train_recall'].append(train_rec)
        training_history['train_f1'].append(train_f1)
        training_history['val_accuracy'].append(val_acc)
        training_history['val_precision'].append(val_prec)
        training_history['val_recall'].append(val_rec)
        training_history['val_f1'].append(val_f1)
        training_history['val_roc_auc'].append(val_roc_auc)
        training_history['val_pr_auc'].append(val_pr_auc)
        training_history['replay_pool_size'].append(int(replay_pool_size))
        training_history['replay_target_size'].append(int(replay_target_size))
        training_history['replay_used_size'].append(int(replay_used_size))

        year_model_path = models_dir / model_file_template.format(year=year_val)
        with open(year_model_path, 'wb') as f:
            pickle.dump(model, f)

        completed_years.add(year_val)
        completion_status['completed_years'][ratio_key] = sorted(list(completed_years))
        all_training_histories[ratio_key] = training_history.copy()
        save_all_training_histories(ratio_histories_file, all_training_histories)
        save_completion_status(ratio_completion_file, completion_status)

        print(
            f'[{ratio_key}] Year {year_val}: Train F1={train_f1:.3f}, Val F1={val_f1:.3f}, Val PR-AUC={val_pr_auc:.3f}, '
            f'replay_used={replay_used_size:,}/{replay_pool_size:,}'
        )
        append_training_log(
            ratio_log_file,
            f'[{ratio_key}] completed year {year_val} with replay_used={replay_used_size}/{replay_pool_size}.',
        )

    finalize_ratio_and_save(
        incremental_scaler, model, models_dir, final_scaler_filename, final_model_filename,
        history_filename, training_history, ratio_key, all_target_year_values, completed_years,
        completion_status, ratio_completion_file, ratio_log_file, all_training_histories, ratio_histories_file,
    )

merged_histories = merge_all_ratio_histories(CHECKPOINT_DIR, ALL_HISTORIES_FILE.stem)
write_combined_history_csv(merged_histories, EXPERIMENTS_DIR / COMBINED_HISTORY_FILENAME)

append_training_log(TRAINING_LOG_FILE, 'Training run completed.')

## 5. Save Training History

In [6]:
import json

CHECKPOINT_DIR = EXPERIMENTS_DIR / 'training_checkpoints_mlp_experience_replay'
ALL_HISTORIES_FILE = CHECKPOINT_DIR / 'mlp_replay_all_training_histories.pkl'
COMPLETION_STATUS_FILE = CHECKPOINT_DIR / 'mlp_replay_completion_status.json'
COMBINED_HISTORY_FILENAME = 'mlp_classifier_history_prevyears_monthly_features_incremental_scaler_experience_replay_all_ratios.csv'

all_training_histories = merge_all_ratio_histories(CHECKPOINT_DIR, ALL_HISTORIES_FILE.stem)

summary_rows = []
for ratio_key in sorted(all_training_histories.keys()):
    history_dict = all_training_histories[ratio_key]
    n_rows = len(history_dict.get('year', []))
    last_year = history_dict['year'][-1] if n_rows > 0 else np.nan
    last_val_f1 = history_dict['val_f1'][-1] if n_rows > 0 else np.nan
    last_val_pr_auc = history_dict['val_pr_auc'][-1] if n_rows > 0 else np.nan
    ratio_completion_status = load_completion_status(per_ratio_path(COMPLETION_STATUS_FILE, ratio_key))
    is_completed = ratio_key in ratio_completion_status.get('completed_ratios', [])
    summary_rows.append(
        {
            'ratio_key': ratio_key,
            'rows': n_rows,
            'last_year': last_year,
            'last_val_f1': last_val_f1,
            'last_val_pr_auc': last_val_pr_auc,
            'completed': is_completed,
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values('ratio_key').reset_index(drop=True)
print('Per-ratio training summary:')
display(summary_df)

combined_history_path = EXPERIMENTS_DIR / COMBINED_HISTORY_FILENAME
if combined_history_path.exists():
    combined_df = pd.read_csv(combined_history_path)
    print(f'Combined history file: {combined_history_path}')
    print(f'Rows: {len(combined_df):,}')
    display(combined_df.tail())
else:
    print(f'Combined history file not found: {combined_history_path}')

Per-ratio training summary:


,ratio_key,rows,last_year,last_val_f1,last_val_pr_auc,completed
0,RR_0.2,6,2022,0.208943,0.354174,True
1,RR_0.3,6,2022,0.197935,0.352706,True
2,RR_0.4,6,2022,0.207466,0.353262,True
3,RR_0.5,6,2022,0.182096,0.346349,True


Combined history file: mlp_classifier_history_prevyears_monthly_features_incremental_scaler_experience_replay_all_ratios.csv
Rows: 24


,year,train_accuracy,train_precision,train_recall,train_f1,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc,val_pr_auc,replay_pool_size,replay_target_size,replay_used_size,ratio_key,replay_ratio
19,2018,0.894251,0.137062,0.783274,0.233300,0.888268,0.134135,0.741875,0.227192,0.902443,0.430126,5575648,2793403,2793403,RR_0.5,0.5
20,2019,0.895399,0.134964,0.769919,0.229668,0.898188,0.123292,0.747838,0.211685,0.904679,0.433421,11162455,2793385,2793385,RR_0.5,0.5
21,2020,0.883286,0.115724,0.792823,0.201968,0.885207,0.102836,0.748954,0.180842,0.903956,0.460669,16749226,2793696,2793696,RR_0.5,0.5
22,2021,0.937515,0.159722,0.728722,0.262015,0.929868,0.143496,0.705095,0.238462,0.904650,0.437929,22336618,2793427,2793427,RR_0.5,0.5
23,2022,0.794728,0.116810,0.868411,0.205921,0.790181,0.102574,0.810227,0.182096,0.882996,0.346349,27923473,2796040,2796040,RR_0.5,0.5
